In [ ]:
# Clean Kaggle environment for Kev + Qwen2.5-0.5B
# Following analysis from copilot-kaggle-debug.md:
# - Delete old numpy/transformers monkey-patching (was fighting Kaggle's 4.49 env)
# - Use coherent Qwen2.5/Transformers/Kev stack instead
# - Pin numpy==2.4.6 + scipy==1.14.1 together (scipy 1.18+ needs numpy 2.5+)
# STEP 1: Pin numpy+scipy TOGETHER before anything else
!pip install -q -U "numpy==2.4.6" "scipy==1.14.1"
# STEP 2: Install ML stack (no sklearn — transformers imports it from scipy)
!pip install -q -U "transformers>=5.17,<6" "peft>=0.21" "datasets>=3.0"
# STEP 3: Install kev WITHOUT deps (it demands numpy>=2.5.3, we need 2.4.6)
!pip install -q --no-deps "git+https://github.com/jaredpalmer/kev.git"
# torchao >=0.16.0 needed by peft LoRA (Kaggle has 0.10.0)
!pip install -q -U "torchao>=0.16.0"

import sys
import torch
import numpy
import transformers
import peft
import kev
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("NumPy:", numpy.__version__)
print("scipy:", scipy.__version__ if "scipy" in dir() else "not imported")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Kev:", kev.__file__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
# Verify Qwen2.5-0.5B loads cleanly (trust_remote_code=False — no custom code needed)
from transformers import AutoTokenizer, AutoModelForCausalLM
BASE = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    dtype=torch.float32,
    attn_implementation="sdpa",
    trust_remote_code=False,
)
print(type(model).__name__)
print("hidden size:", model.config.hidden_size)
print("model type:", model.config.model_type)
del model
torch.cuda.empty_cache()
print("Model load test passed!", )


In [ ]:
# Convert WCAG data from old format to current Kev format
import json, sys, subprocess, random
from pathlib import Path
src = Path("/kaggle/input/wcag-kev-training/wcag_train.jsonl")
dst = Path("/kaggle/working/wcag_train_kev.jsonl")
if not src.exists():
    print("Dataset not found at", src, )
    print("Generating 200 WCAG patterns inline...", )
    patterns = [
        ("Image without alt", '<img src="photo.jpg">', True, ["1.1.1"], 4),
        ("Image empty alt", '<img src="dec.png" alt="">', False, [], 0),
        ("Image with alt", '<img src="chart.png" alt="Sales">', False, [], 0),
        ("Missing label", '<input type="text" name="x">', True, ["1.3.1","3.3.2"], 4),
        ("Has label", '<label>Email</label><input name="x">', False, [], 0),
        ("Empty heading", '<h2></h2>', True, ["1.3.1"], 2),
        ("Good heading", '<h2>Title</h2>', False, [], 0),
        ("Link no text", '<a href="/p"></a>', True, ["2.4.4","4.1.2"], 3),
        ("Link text", '<a href="/p">Link</a>', False, [], 0),
        ("Button no name", '<button></button>', True, ["4.1.2"], 3),
        ("Button text", '<button>Submit</button>', False, [], 0),
        ("Missing th", '<table><tr><td>D</td></tr></table>', True, ["1.3.1"], 3),
        ("Has th", '<table><thead><tr><th>C</th></tr></thead><tbody><tr><td>D</td></tr></tbody></table>', False, [], 0),
        ("Low contrast", '<p style="color:#ccc">T</p>', True, ["1.4.3"], 3),
        ("Good contrast", '<p style="color:#000">T</p>', False, [], 0),
        ("Missing lang", '<html><body>C</body></html>', True, ["3.1.1"], 2),
        ("Has lang", '<html lang="en"><body>C</body></html>', False, [], 0),
        ("No landmark", '<div>Main</div>', True, ["1.3.1"], 2),
        ("Has landmark", '<nav aria-label="Main">N</nav>', False, [], 0),
        ("Focus hidden", '<a href="#" style="outline:none">L</a>', True, ["2.4.7"], 2),
    ]
    rows = []
    for i in range(200):
        p = random.choice(patterns)
        label_idx = 0 if not p[3] else p[3].index(p[3][0])
        rows.append({
            "state": f"WCAG pattern {i+1}",
            "questions": [
                {"type": "noul", "instr": f'Does "{p[1]}" have a violation?', "label": p[2]},
                {"type": "choice", "instr": "Which WCAG criterion?", "options": p[3] + (["None"] if not p[3] else []), "label": label_idx},
                {"type": "score", "instr": "Severity 0-4", "options": ["0","1","2","3","4"], "label": p[4]},
            ],
            "_meta": {"pattern": p[0]}
        })
    with dst.open("w", encoding="utf-8") as fout:
        for record in rows:
            questions = {}
            for qi, q in enumerate(record["questions"]):
                if q["type"] == "noul":
                    key = "has_violation" if qi == 0 else "is_accessible"
                    questions[key] = {"type": "noul", "instructions": q["instr"], "label": bool(q["label"])}
                elif q["type"] == "choice":
                    questions["which_criterion"] = {
                        "type": "choice",
                        "instructions": q["instr"],
                        "criteria": {o: None for o in q["options"]},
                        "label": q["options"][q["label"]]
                    }
                elif q["type"] == "score":
                    questions["severity"] = {
                        "type": "score",
                        "instructions": q["instr"],
                        "criteria": q["options"],
                        "label": int(q["label"])
                    }
            converted = {"state": record["state"], "questions": questions, "_meta": record.get("_meta", {})}
            fout.write(json.dumps(converted, ensure_ascii=False) + "\n")
    print(f"Generated {len(rows)} records -> {dst}", )
else:
    print(f"Found dataset at {src}", )
    with src.open(encoding="utf-8") as fin, dst.open("w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip():
                continue
            record = json.loads(line)
            questions = {}
            for i, q in enumerate(record["questions"]):
                if q["type"] == "noul":
                    key = "has_violation" if i == 0 else "is_accessible"
                    questions[key] = {
                        "type": "noul",
                        "instructions": q["instr"],
                        "label": bool(q["label"])
                    }
                elif q["type"] == "choice":
                    questions["which_criterion"] = {
                        "type": "choice",
                        "instructions": q["instr"],
                        "criteria": {opt: None for opt in q["options"]},
                        "label": q["options"][q["label"]]
                    }
                elif q["type"] == "score":
                    questions["severity"] = {
                        "type": "score",
                        "instructions": q["instr"],
                        "criteria": q["options"],
                        "label": int(q["label"])
                    }
            converted = {
                "state": record["state"],
                "questions": questions,
                "_meta": record.get("_meta", {})
            }
            fout.write(json.dumps(converted, ensure_ascii=False) + "\n")
    print(f"Converted {src} -> {dst}")
print(f"Output: {dst}")
print(f"Lines: {sum(1 for _ in open(dst))}")


In [ ]:
# Train Kev-WCAG using upstream kev.train CLI
import subprocess
import sys
DATA = "/kaggle/working/wcag_train_kev.jsonl"
OUT = "/kaggle/working/kev-wcag"
cmd = [
    sys.executable, "-m", "kev.train",
    "--data", DATA,
    "--base", "Qwen/Qwen2.5-0.5B",
    "--epochs", "3",
    "--lr", "2e-4",
    "--lora", "16",
    "--batch", "1",
    "--accum", "8",
    "--dtype", "fp32",
    "--weights_dtype", "fp32",
    "--device", "cuda",
    "--out", OUT,
]
print("Running:", " ".join(cmd), )
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout, )
if result.stderr:
    print(result.stderr, )
print(f"Exit code: {result.returncode}", )
from pathlib import Path
for f in sorted(Path(OUT).glob("**/*")):
    if f.is_file():
        print(f"  {f.relative_to(OUT)}: {f.stat().st_size/1024:.1f} KB", )


In [ ]:
# Save trained model to output for Kaggle dataset export
import shutil
from pathlib import Path
SRC = "/kaggle/working/kev-wcag"
DST = "/kaggle/working/kev-wcag-model"
if Path(SRC).exists():
    shutil.copytree(SRC, DST, dirs_exist_ok=True)
    print(f"Model copied to {DST}")
else:
    print(f"Source {SRC} not found - training may not have completed")
print("\nOutput files:")
for f in sorted(Path(DST).glob("**/*")):
    if f.is_file():
        print(f"  {f.relative_to(DST)}: {f.stat().st_size/1024:.1f} KB")
print("\nDONE!", )
